# Task 1 — Sentiment Analysis

Train and compare a TF-IDF baseline against a fine-tuned transformer. Report accuracy, per-class precision/recall/F1 and the confusion matrix on a held-out test set. Save the chosen model to `models/`.

In [14]:
import sys
sys.path.append("..")

import importlib
import re
import string

import matplotlib.pyplot as plt
import pandas as pd
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from src import config, preprocessing

importlib.reload(preprocessing)

pre_processed = preprocessing.load_pre_processed()
pre_processed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27863 entries, 0 to 27862
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            27863 non-null  object 
 1   categories      27863 non-null  object 
 2   reviews.rating  27863 non-null  float64
 3   reviews.text    27863 non-null  object 
 4   reviews.title   27863 non-null  object 
dtypes: float64(1), object(4)
memory usage: 1.1+ MB


In [15]:
rating_to_sentiment = {
    1.0: "bad",
    2.0: "bad",
    3.0: "neutral",
    4.0: "good",
    5.0: "good",
}
y = pre_processed["reviews.rating"].map(rating_to_sentiment)
X = pre_processed["reviews.text"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y,
)
# X_train, X_test, y_train, y_test = train_test_split(pre_processed['reviews.text'], pre_processed['reviews.rating'], test_size=0.2, random_state=42)

print(X_train.shape, y_train.shape)          # (22290,) (22290,)
print(X_train.value_counts(normalize=True))
print(y_train.value_counts(normalize=True))
print(type(X_train))

(19504,) (19504,)
reviews.text
Product is awesome. Amazing how I ask a question and in a few seconds she has the answer.                                                                                                                                                                                        0.000051
Brought for grandmother as a Christmas gift because she wanted and iPad and we. Thought she'd never figure out how to use one so I bought this as a beginners tablet and she loves it                                                                                            0.000051
Great tablet for the price. Add to that expandable memory and it's a great gift for the holidays.                                                                                                                                                                                0.000051
This is a great tables for browsing the net, reading or listening to music. However when moving to video apps like YouTube,

In [16]:
import re

# Remove inline JS/CSS first
# Then remove comments before tags, because comments may contain '>'
# Finally remove the remaining HTML tags

def remove_html_tags(text):
    text = re.sub(r"<script.*?</script>", " ", text, flags=re.I | re.S)
    text = re.sub(r"<style.*?</style>", " ", text, flags=re.I | re.S)
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.I | re.S)
    text = re.sub(r"<[^>]+>", " ", text)
    return text

# Apply to the text column
X_train_cleaned = X_train.apply(remove_html_tags)
X_test_cleaned = X_test.apply(remove_html_tags)

In [17]:
# Remove special characters, numbers, and extra spacing
# Remove single-letter words and prefixed 'b'
# Convert to lowercase

def clean_text(text):
    if not isinstance(text, str):
        text = str(text)

    text = text.lower()
    #text = text.replace("b'", "").replace('b"', "")
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\b[a-z]\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

X_train_cleaned = X_train_cleaned.apply(clean_text)
X_test_cleaned = X_test_cleaned.apply(clean_text)

X_train_cleaned.head()

26559    product is awesome amazing how ask question an...
18910    love the kindle paperwhite it is so light weig...
8037     great for kids easy to use great price would r...
4539     this was great purchase love the size and func...
8332     it is great for reading but that about it ther...
Name: reviews.text, dtype: object

In [18]:
def remove_stopwords(text):
    stop_words = set(stopwords.words("english"))
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return " ".join(filtered_words)

X_train_cleaned = X_train_cleaned.apply(remove_stopwords)
X_test_cleaned = X_test_cleaned.apply(remove_stopwords)

X_train_cleaned.head(10)

26559    product awesome amazing ask question seconds a...
18910    love kindle paperwhite light weight best read ...
8037       great kids easy use great price would recommend
4539                great purchase love size functionality
8332     great reading ads quite annoying well slow spe...
14375    looking good cheap tablet general use fit bill...
7664     purchased gift christmas girlfriend gave extra...
1040     avid amazon user shopping tablets bought diffe...
597      thing looks nice runs nice feels nice price th...
2312                    lot cheaper tablets functions need
Name: reviews.text, dtype: object

In [19]:
# Your code
from nltk.stem.wordnet import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
X_train_lemmatized = X_train_cleaned.apply(lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()]))
X_test_lemmatized = X_test_cleaned.apply(lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()]))

In [20]:
# Convert reviews to TF-IDF features.
from sklearn.feature_extraction.text import TfidfVectorizer


tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train_lemmatized)
X_test_tfidf = tfidf.transform(X_test_lemmatized)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)


def top_tfidf_terms_for_ratings(categories, labels, tfidf_matrix, feature_names, top_n=10):
    """Return terms with the highest average TF-IDF for selected labels."""
    category_rows = labels.isin(categories).to_numpy()
    if not category_rows.any():
        raise ValueError(f"No training rows found for labels {categories}; check y_train values.")
    mean_scores = tfidf_matrix[category_rows].mean(axis=0).A1
    return pd.Series(mean_scores, index=feature_names).nlargest(top_n)


feature_names = tfidf.get_feature_names_out()
labels = y_train.reset_index(drop=True)

negative = top_tfidf_terms_for_ratings(["bad"], labels, X_train_tfidf, feature_names)
neutral = top_tfidf_terms_for_ratings(["neutral"], labels, X_train_tfidf, feature_names)
positive = top_tfidf_terms_for_ratings(["good"], labels, X_train_tfidf, feature_names)

print("\nTop negative TF-IDF terms:\n", negative.head())
print("\nTop neutral TF-IDF terms:\n", neutral.head())
print("\nTop positive TF-IDF terms:\n", positive.head())

Train TF-IDF shape: (19504, 5000)
Test TF-IDF shape: (8359, 5000)

Top negative TF-IDF terms:
 tablet    0.040975
amazon    0.038305
slow      0.032070
work      0.029584
buy       0.028488
dtype: float64

Top neutral TF-IDF terms:
 tablet    0.055714
good      0.044583
price     0.033156
like      0.029135
apps      0.026756
dtype: float64

Top positive TF-IDF terms:
 great     0.053797
love      0.050576
tablet    0.049604
use       0.036796
easy      0.035177
dtype: float64


In [21]:
# Convert reviews to TF-IDF features.
from sklearn.feature_extraction.text import TfidfVectorizer


tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=20000,
    sublinear_tf=True,
)
X_train_tfidf = tfidf.fit_transform(X_train_lemmatized)
X_test_tfidf = tfidf.transform(X_test_lemmatized)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)


def top_tfidf_terms_for_ratings(categories, labels, tfidf_matrix, feature_names, top_n=10):
    """Return terms with the highest average TF-IDF for selected labels."""
    category_rows = labels.isin(categories).to_numpy()
    if not category_rows.any():
        raise ValueError(f"No training rows found for labels {categories}; check y_train values.")
    mean_scores = tfidf_matrix[category_rows].mean(axis=0).A1
    return pd.Series(mean_scores, index=feature_names).nlargest(top_n)


feature_names = tfidf.get_feature_names_out()
labels = y_train.reset_index(drop=True)

negative = top_tfidf_terms_for_ratings(["bad"], labels, X_train_tfidf, feature_names)
neutral = top_tfidf_terms_for_ratings(["neutral"], labels, X_train_tfidf, feature_names)
positive = top_tfidf_terms_for_ratings(["good"], labels, X_train_tfidf, feature_names)

print("\nTop negative TF-IDF terms:\n", negative.head())
print("\nTop neutral TF-IDF terms:\n", neutral.head())
print("\nTop positive TF-IDF terms:\n", positive.head())

Train TF-IDF shape: (19504, 20000)
Test TF-IDF shape: (8359, 20000)

Top negative TF-IDF terms:
 amazon      0.025565
tablet      0.024714
slow        0.023149
returned    0.019698
work        0.019071
dtype: float64

Top neutral TF-IDF terms:
 tablet    0.033078
good      0.027631
price     0.021269
like      0.018453
ok        0.017315
dtype: float64

Top positive TF-IDF terms:
 great     0.030453
love      0.028384
tablet    0.027558
use       0.021026
easy      0.020141
dtype: float64


In [22]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

class_order = ["bad", "neutral", "good"]
nb_model = MultinomialNB()
nb_sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
nb_model.fit(X_train_tfidf, y_train, sample_weight=nb_sample_weights)

y_pred_nb = nb_model.predict(X_test_tfidf)

print("Multinomial Naive Bayes")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb, labels=class_order, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_nb, labels=class_order))

Multinomial Naive Bayes
Accuracy: 0.7834669218806077
              precision    recall  f1-score   support

         bad       0.18      0.57      0.27       199
     neutral       0.15      0.53      0.23       387
        good       0.98      0.80      0.88      7773

    accuracy                           0.78      8359
   macro avg       0.44      0.63      0.46      8359
weighted avg       0.92      0.78      0.84      8359

Confusion matrix:
 [[ 113   62   24]
 [  91  204   92]
 [ 429 1112 6232]]


In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

classifier = LogisticRegression(class_weight="balanced", max_iter=1000)
classifier.fit(X_train_tfidf, y_train)

y_pred = classifier.predict(X_test_tfidf)

print(classification_report(y_test, y_pred, labels=["bad", "neutral", "good"], zero_division=0))
print(confusion_matrix(y_test, y_pred, labels=["bad", "neutral", "good"]))

              precision    recall  f1-score   support

         bad       0.31      0.46      0.37       199
     neutral       0.22      0.45      0.30       387
        good       0.97      0.91      0.94      7773

    accuracy                           0.88      8359
   macro avg       0.50      0.61      0.54      8359
weighted avg       0.92      0.88      0.90      8359

[[  92   53   54]
 [  50  173  164]
 [ 155  544 7074]]


In [24]:
from sklearn.svm import LinearSVC

model = LinearSVC(class_weight="balanced")
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

class_order = ["bad", "neutral", "good"]

print("Accuracy:", accuracy_score(y_test, y_pred))
print(
    classification_report(
        y_test,
        y_pred,
        labels=class_order,
        zero_division=0,
    )
)
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred, labels=class_order))

Accuracy: 0.924273238425649
              precision    recall  f1-score   support

         bad       0.42      0.24      0.31       199
     neutral       0.34      0.26      0.29       387
        good       0.95      0.98      0.96      7773

    accuracy                           0.92      8359
   macro avg       0.57      0.49      0.52      8359
weighted avg       0.91      0.92      0.92      8359

Confusion matrix:
[[  48   36  115]
 [  27   99  261]
 [  40  154 7579]]


## Improving the model

Testing the first saved model in the app showed it handles positive reviews well but fails on negative and neutral ones
(e.g. *"Worst customer service I have ever experienced"* → positive). Reasons:

1. **Very few negative / neutral examples**: 93% of reviews are positive, so words like "price", "quality" and "customer service" mostly appear in positive reviews.
2. **Cleaning removed key words**: the NLTK stopword list drops "not", "no", "too", "very", so *"price is too high"* became *"price high"*.
3. **TF-IDF only counts words**, it does not understand meaning.

What we try, all on the **same test split** (`test_size=0.3`, `random_state=42`, stratified):

| Model | Change |
|---|---|
| TF-IDF v1 (cells above) | stopwords + lemmatizing, text only → macro-F1 ≈ 0.54 (Logistic Regression) |
| **TF-IDF v2** | keep negation words, expand "n't" → "not", no lemmatizing, **title + text**, `C=10` |
| **Pretrained transformer (zero-shot)** | `cardiffnlp/twitter-roberta-base-sentiment-latest`, already trained for negative/neutral/positive, no training on our data |

We compare by **macro-F1** (accuracy hides the small classes) and by a few hand-written reviews.

*Roman Urdu: Har model ko same test set pe macro-F1 se compare karo, phir jeetne wala app mein jayega.*

In [25]:
import sys
sys.path.append("../app")

import json
import joblib
import nltk
import sklearn
import numpy as np
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.model_selection import train_test_split

import importlib, text_cleaning
importlib.reload(text_cleaning)          # pick up edits to app/text_cleaning.py
from text_cleaning import preprocess

LABELS = ["negative", "neutral", "positive"]
APP_DIR = Path("../app")

# Title + text: titles like "Terrible" or "Disappointed" carry a lot of sentiment
X_full = pre_processed["reviews.title"].fillna("").astype(str) + ". " + pre_processed["reviews.text"].astype(str)
y_named = pre_processed["reviews.rating"].map({1.0: "negative", 2.0: "negative", 3.0: "neutral", 4.0: "positive", 5.0: "positive"})

X_tr, X_te, y_tr, y_te = train_test_split(X_full, y_named, test_size=0.3, random_state=42, stratify=y_named)

# Hand-written reviews, including the ones the first app got wrong
HAND_TESTS = [
    "price is too high and quality is really bad",
    "This product broke after two days of light use.",
    "Worst customer service I have ever experienced.",
    "Terrible interface, nothing works as advertised",
    "not going to recommend to anyone",
    "It arrived damaged and missing key parts",
    "It is a standard blue shirt made of cotton",
    "It works, but the screen is dimmer than I expected for the price.",
    "Exceeded my expectations, absolutely love it!",
]

results = {}      # model name -> scores, filled below

def evaluate(name, y_true, y_pred):
    f1 = f1_score(y_true, y_pred, labels=LABELS, average=None)
    results[name] = {
        "accuracy": round(accuracy_score(y_true, y_pred), 3),
        "macro_f1": round(f1.mean(), 3),
        "f1_negative": round(f1[0], 3),
        "f1_neutral": round(f1[1], 3),
        "f1_positive": round(f1[2], 3),
    }
    print(f"=== {name}")
    print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0))
    print(confusion_matrix(y_true, y_pred, labels=LABELS))

### TF-IDF v2

In [26]:
tfidf_v2 = Pipeline([
    ("clean", FunctionTransformer(preprocess)),
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)),
    ("model", LogisticRegression(class_weight="balanced", C=10, max_iter=2000)),
])
tfidf_v2.fit(X_tr, y_tr)                 # raw text in; the Pipeline cleans it itself
evaluate("TF-IDF v2 + LogReg", y_te, tfidf_v2.predict(X_te))

=== TF-IDF v2 + LogReg
              precision    recall  f1-score   support

    negative       0.48      0.51      0.50       199
     neutral       0.35      0.36      0.36       387
    positive       0.97      0.96      0.97      7773

    accuracy                           0.93      8359
   macro avg       0.60      0.61      0.61      8359
weighted avg       0.93      0.93      0.93      8359

[[ 102   44   53]
 [  50  141  196]
 [  61  222 7490]]


### Pretrained transformer (zero-shot)
No training: the model was already trained on ~124M tweets for negative / neutral / positive.
Needs `transformers` and `torch`: if the import fails, run `%pip install transformers torch` and restart the kernel.
The first run downloads the model (~500 MB). Predicting the 8,359 test reviews takes a few minutes on a Mac;
set `EVAL_SAMPLE = 2000` for a quicker (approximate) check.

In [27]:
import torch
from transformers import pipeline as hf_pipeline

TRANSFORMER_ID = "cardiffnlp/twitter-roberta-base-sentiment-latest"
EVAL_SAMPLE = None          # e.g. 2000 for a faster check; None = full test set

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
transformer = hf_pipeline("text-classification", model=TRANSFORMER_ID, device=device)

X_eval, y_eval = X_te, y_te
if EVAL_SAMPLE:
    X_eval = X_te.sample(EVAL_SAMPLE, random_state=42)
    y_eval = y_te.loc[X_eval.index]

out = transformer(X_eval.tolist(), batch_size=32, truncation=True, max_length=512)
y_pred_tr = [d["label"].lower() for d in out]
evaluate("Transformer zero-shot (roberta)", y_eval, y_pred_tr)

if EVAL_SAMPLE:   # score TF-IDF v2 on the same subset, so the comparison is fair
    evaluate("TF-IDF v2 + LogReg (same sample)", y_eval, tfidf_v2.predict(X_eval))

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

=== Transformer zero-shot (roberta)
              precision    recall  f1-score   support

    negative       0.43      0.80      0.56       199
     neutral       0.24      0.11      0.15       387
    positive       0.97      0.97      0.97      7773

    accuracy                           0.93      8359
   macro avg       0.55      0.63      0.56      8359
weighted avg       0.92      0.93      0.92      8359

[[ 160   17   22]
 [ 119   44  224]
 [  94  121 7558]]


### Compare

In [28]:
comparison = pd.DataFrame(results).T.sort_values("macro_f1", ascending=False)
display(comparison)

def tfidf_label(text):
    return tfidf_v2.predict([text])[0]

def transformer_label(text):
    return transformer(text, truncation=True, max_length=512)[0]["label"].lower()

hand = pd.DataFrame({
    "review": HAND_TESTS,
    "TF-IDF v2": [tfidf_label(t) for t in HAND_TESTS],
    "Transformer": [transformer_label(t) for t in HAND_TESTS],
})
display(hand)

,accuracy,macro_f1,f1_negative,f1_neutral,f1_positive
TF-IDF v2 + LogReg,0.925,0.605,0.495,0.355,0.966
Transformer zero-shot (roberta),0.929,0.562,0.559,0.155,0.970


,review,TF-IDF v2,Transformer
0,price is too high and quality is really bad,positive,negative
1,This product broke after two days of light use.,negative,negative
2,Worst customer service I have ever experienced.,positive,negative
3,"Terrible interface, nothing works as advertised",neutral,negative
4,not going to recommend to anyone,positive,negative
5,It arrived damaged and missing key parts,negative,negative
6,It is a standard blue shirt made of cotton,neutral,neutral
7,"It works, but the screen is dimmer than I expe...",neutral,neutral
8,"Exceeded my expectations, absolutely love it!",positive,positive


### Choose and save the model for the app
By default the model with the higher macro-F1 wins. Look at the hand-written tests too:
if the scores are close, prefer the model that gets those right, and set `CHOICE` yourself.
Write one sentence on why you chose it.

The cell writes `app/model_config.json` (which model the app loads) and `app/requirements.txt` (with your exact versions).

In [ ]:
CHOICE = "transformer"    # None = pick by macro-F1, or set "tfidf" / "transformer"

tfidf_name = "TF-IDF v2 + LogReg (same sample)" if EVAL_SAMPLE else "TF-IDF v2 + LogReg"
if CHOICE is None:
    CHOICE = "transformer" if results["Transformer zero-shot (roberta)"]["macro_f1"] > results[tfidf_name]["macro_f1"] else "tfidf"
print("Serving:", CHOICE)

# The TF-IDF model is always saved (small, useful as a fallback)
joblib.dump(tfidf_v2, APP_DIR / "sentiment_model.joblib")
joblib.dump(tfidf_v2, config.MODELS_DIR / "sentiment_tfidf_v2_logreg.joblib")

if CHOICE == "transformer":
    import transformers
    model_config = {"type": "transformer", "model_id": TRANSFORMER_ID}
    requirements = f"gradio\ntransformers=={transformers.__version__}\ntorch\n"
else:
    model_config = {"type": "tfidf"}
    requirements = (
        "gradio\n"
        f"scikit-learn=={sklearn.__version__}\n"
        f"nltk=={nltk.__version__}\n"
        f"joblib=={joblib.__version__}\n"
    )

(APP_DIR / "model_config.json").write_text(json.dumps(model_config))
(APP_DIR / "requirements.txt").write_text(requirements)
comparison.to_csv(config.METRICS_DIR / "sentiment_model_comparison.csv")

print("model_config.json:", model_config)
print("requirements.txt:\n" + requirements)

Serving: tfidf
model_config.json: {'type': 'tfidf'}
requirements.txt:
gradio
scikit-learn==1.7.2
nltk==3.9.2
joblib==1.5.2

